## Importar o CSV

In [37]:
import pandas as pd

cols = ['NU_INSCRICAO', 'IN_TREINEIRO', 'TP_SEXO', 'TP_FAIXA_ETARIA',
        'TP_COR_RACA', 'TP_ESTADO_CIVIL', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU',
        'TP_ENSINO', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA',
        'CO_UF_PROVA', 'SG_UF_PROVA', 'Q001', 'Q002', 'Q003', 'Q004',
        'Q005', 'Q006', 'Q020', 'Q023']

df = pd.read_csv(
    '/home/gabriel/Documentos/AnaliseDadosBigData/microdados_enem_2025/DADOS/PARTICIPANTES_2025.csv',
    sep=';',
    encoding='ISO-8859-1',
    usecols=cols
)


In [38]:
df

,NU_INSCRICAO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_PROVA,...,CO_UF_PROVA,SG_UF_PROVA,Q001,Q002,Q003,Q004,Q005,Q006,Q020,Q023
0,210066506229,6,F,1,2,1,4,NaN,0,2932200,...,29,BA,C,F,A,A,1,B,A,A
1,210066506230,3,M,1,1,2,0,1.0,0,2910800,...,29,BA,B,E,C,B,3,A,B,A
2,210066506231,2,F,1,1,3,0,NaN,1,3549805,...,35,SP,E,E,D,D,6,A,B,C
3,210066506232,2,M,1,3,2,0,1.0,0,2203909,...,22,PI,D,F,D,D,3,A,B,D
4,210066506233,1,F,1,1,3,0,NaN,1,5101803,...,51,MT,G,G,E,E,4,B,B,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4810767,210071444509,12,F,1,1,1,14,NaN,0,2611606,...,26,PE,G,G,E,E,7,B,B,D
4810768,210071444513,4,M,1,1,1,1,NaN,0,3522208,...,35,SP,F,E,D,B,4,A,B,E
4810769,210071444514,11,M,1,1,1,10,NaN,0,3505708,...,35,SP,E,E,F,F,4,B,B,A
4810770,210071444515,8,F,1,2,1,5,NaN,0,3509502,...,35,SP,E,E,F,F,1,B,B,A


---

## Filtrar para trocar os Nulos e Treineiros

In [39]:
df['IN_TREINEIRO'].value_counts() #Saber quantos tem
df = df[df['IN_TREINEIRO'] == 0].copy() #Filtra só para não ter treineiros


In [53]:
df.isnull().sum().sort_values(ascending=False) #Ver os nulos por coluna do maior ao menor
#Somento o TP_ENSINO ta(tava) com 2092958 valores nulos

NU_INSCRICAO          0
TP_FAIXA_ETARIA       0
TP_SEXO               0
TP_ESTADO_CIVIL       0
TP_COR_RACA           0
TP_ST_CONCLUSAO       0
TP_ANO_CONCLUIU       0
TP_ENSINO             0
IN_TREINEIRO          0
CO_MUNICIPIO_PROVA    0
NO_MUNICIPIO_PROVA    0
CO_UF_PROVA           0
SG_UF_PROVA           0
Q001                  0
Q002                  0
Q003                  0
Q004                  0
Q005                  0
Q006                  0
Q020                  0
Q023                  0
dtype: int64

In [41]:
#Colunas de identifição e localização
cols_essenciais = ['CO_MUNICIPIO_PROVA', "SG_UF_PROVA", "TP_SEXO"]
df = df.dropna(subset=cols_essenciais)

#Colunas de questionario socioeconomico
cols_questionario = ['Q001', "Q002", "Q003", "Q004", "Q005", "Q006"]
for col in cols_questionario:
    df[col] = df[col].fillna("NÃO INFORMADO")

#Categorias das caracteristicas do candiadato
cols_tp = ['TP_COR_RACA', 'TP_ESTADO_CIVIL', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU', 'TP_ENSINO']
for col in cols_tp:
    df[col] = df[col].fillna('NÃO INFORMADO')

---

## Participação por município/UF

In [49]:
part_municipio = (
    df.groupby(["NO_MUNICIPIO_PROVA", "SG_UF_PROVA"]).size()
    .reset_index(name='qtd_participantes')
    .sort_values('qtd_participantes' , ascending=False)
)

part_municipio.head(10)


,NO_MUNICIPIO_PROVA,SG_UF_PROVA,qtd_participantes
1610,São Paulo,SP,160867
1383,Rio de Janeiro,RJ,110561
1404,Salvador,BA,72167
260,Brasília,DF,67665
603,Fortaleza,CE,65840
212,Belo Horizonte,MG,59170
949,Manaus,AM,57916
215,Belém,PA,56554
1593,São Luís,MA,45548
1345,Recife,PE,42009


In [ ]:
"""
Pensando em inisght pra cá, talvez junte com o Urbano X Rural para enriquecer mais aqui
"""
part_municipio.tail(10)

,NO_MUNICIPIO_PROVA,SG_UF_PROVA,qtd_participantes
1715,Uiramutã,RR,104
392,Carlinda,MT,101
95,Araguanã,TO,101
184,Barra do Turvo,SP,99
1147,Pacaraima,RR,89
264,Brejinho de Nazaré,TO,70
51,Amajari,RR,53
1440,Santa Rosa do Purus,AC,48
588,Fernando de Noronha,PE,44
587,Fernando Falcão,MA,32


In [51]:
part_uf = df["SG_UF_PROVA"].value_counts().reset_index()
part_uf.columns = ["UF", "qtd_participantes"]

part_uf.head(5)

,UF,qtd_participantes
0,SP,603948
1,MG,368958
2,BA,342861
3,RJ,275221
4,PA,228700


In [ ]:
part_uf.tail(5)

,UF,qtd_participantes
22,RO,38576
23,TO,27323
24,AP,27263
25,AC,24447
26,RR,11212


In [ ]:
#Grafico com matplos

---


## Candidato Possui internet?

In [ ]:
"""
De acordo com o Dicionario do enem (arquivo esse na pasta /Dicionario/Dicionário_Microdados_Enem_2025.xlsx)
A - Não
B - Sim

87,62% dos candidatos possuem internet
"""

df['Q020'].value_counts(normalize=True) * 100

Q020
B    87.626709
A    12.373291
Name: proportion, dtype: float64

In [ ]:
#Vamos separar por estado

"""
É notado uma grande deficiencia com internet na nossa nação, observando na tabela onde o estado
do Amazonas , os candidatos que possuem não internet ocupam 31.22% do total de inscritos, isso sem
incluir treineiros
"""

tabela_internet_uf = pd.crosstab(df['SG_UF_PROVA'], df['Q020'], normalize='index') * 100
tabela_internet_uf.round(2).sort_values('A', ascending=False)



Q020,A,B
SG_UF_PROVA,,
AM,31.22,68.78
PA,27.04,72.96
AC,27.03,72.97
MA,23.11,76.89
AP,21.90,78.10
PI,18.30,81.70
SE,17.75,82.25
CE,16.55,83.45
TO,16.03,83.97


In [ ]:
#Grafico com matplos